In [38]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [39]:
data = pd.read_csv('../data/simpsons_script_lines.csv')

# 1. Filtering Speaking Lines

we detect that speaking line should be a boolean, but it's not as we have lower case intances, correct the dtype 

In [40]:
data['speaking_line'] = data['speaking_line'].replace({'false': False, 'true': True}).astype(bool)

we detect that instances with speaking line = false refer to two narration, sounds of objects of the show (no character associated) or onomatopeias from the characters.
This instances won't hlp us analyzing the words of the characters and therefore we decide to delete them from the dataset to reduce it's size and keep only relevenat instances.
Since the column will be a constant after this filter, we delete it too.


In [41]:
data = data[data['speaking_line']]
data.drop(columns=['speaking_line'], inplace=True)

# 2 Incorrect Word Counts

we notice thah there are 10 incorrect rows, where the spoken words have a different format, the normalized_text is not a sequence of word, but an integer abd the word_count is true instead of an integer.
Most of this lines belong to non-main characters ot groups like 'Entire Town', singers or others.
There are a few of them that belong to the singer tonny bennet. Even though the character exists these and appears to say :Hey, good to see you these lines belong to the background song of the epsiode(https://www.youtube.com/watch?v=dZsOHz1z7Pc). Since tony bennet only says one line and the other ones are not conversation lines, we decide to rease all instances of tonny bennet.
We delte the instances from 'entire town', singers and Revue Cast Members, too, since they don't belong to individual characters and have very little represntation on the entirity of the dataset.


Moe Szyslak: "The reason I left you is simple:

In [42]:
data[data['word_count'] == 'true']

,id,episode_id,number,raw_text,timestamp_in_ms,character_id,location_id,raw_character_text,raw_location_text,spoken_words,normalized_text,word_count
8082,17667,59,87,"Singers: (SINGING) ""IT'S THE FIRST ANNUAL MONT...",442000,276.0,636.0,Singers,Springfield Civic Center,IT'S THE FIRST ANNUAL MONTGOMERY BURNS/ AWARD ...,455000,true
71799,81724,282,279,"Revue Cast Members: ""WE'RE THE PERFORMERS YOU ...",1068000,3510.0,2370.0,Revue Cast Members,Branson Theater,WE'RE THE PERFORMERS YOU THOUGHT WERE DEAD / L...,1075000,true
81136,91095,315,284,"Homer Simpson: ""I-M-O--",1220000,2.0,5.0,Homer Simpson,Simpson Home,"""I-M-O--,i-m-o--,1\n91096,315,285,Homer Simpso...",1220000,true
115436,125627,445,216,"Moe Szyslak: ""The reason I left you is simple:",1095000,17.0,15.0,Moe Szyslak,Moe's Tavern,"""The reason I left you is simple:,the reason i...",1095000,true
153015,4260,14,252,"Entire Town: ""SLEIGH BELLS RING / ARE YOU LIST...",1121000,241,211.0,Entire Town,NEIGHBORHOOD,"""SLEIGH BELLS RING / ARE YOU LISTENIN'/,sleigh...",1134000,true
154078,5345,18,199,"Tony Bennett: ""THERE'S A SWINGIN' TOWN I KNOW /",993000,297,239.0,Tony Bennett,Capital City,"THERE'S A SWINGIN' TOWN I KNOW /,theres a swin...",997000,true
154080,5348,18,202,"Tony Bennett: ""PEOPLE STOP AND SCREAM HELLO /",1001000,297,239.0,Tony Bennett,Capital City,"PEOPLE STOP AND SCREAM HELLO /,people stop and...",1004000,true
154082,5351,18,205,"Tony Bennett: ""IT'S THE KIND OF PLACE THAT MAKES",1009000,297,239.0,Tony Bennett,Capital City,"IT'S THE KIND OF PLACE THAT MAKES,its the kind...",1009000,true
154084,5354,18,208,"Tony Bennett: ""AND IT MAKES A KING FEEL LIKE/",1016000,297,239.0,Tony Bennett,Capital City,"AND IT MAKES A KING FEEL LIKE/,and it makes a ...",1019000,true
156870,8152,28,13,"Gulliver Dark: (SINGING) ""THE RULES THAT CONST...",115000,187,332.0,Gulliver Dark,Aztec Theater,THE RULES THAT CONSTRAIN OTHER MEN / MEAN NOTH...,125000,true


In [43]:
data = data[~data['raw_character_text'].isin(['Tony Bennett','Singers','Revue Cast Members','Entire Town'])]

This error comes from the transcription of the sentcne I am okay as initials I.M.O.K (Disney's subtitles also show the same)
This is actually the end of the previous sentence and not a new one. We will delete this sentence and add the correct normalized ending to the previous one and delete the incorrect one.

Also the sentence Homer says afterwards is lost. He says_  'get it? I am okay'. We will assing this sentence to the current incorrect one

In [44]:
current_sentence = 'last weeks garbage i missed the pick-up date but it doesnt matter because my mom is alive see'
missing_part = 'i am okay'
correct_sentence = current_sentence + ' ' + missing_part
word_count = len(correct_sentence.split())
data.loc[data["id"] == 91094, "word_count"] = word_count
data.loc[data["id"] == 91094, "normalized_text"] = correct_sentence

lost_sentence = 'get it i am okay'
word_count = len(lost_sentence.split())
data.loc[data["id"] == 91095, "word_count"] = word_count
data.loc[data["id"] == 91095, "normalized_text"] = lost_sentence

Moe's sentence error is also fixable, there is a missing part. 
The correct sentence is :  "The reason I left you is simple: I'm gay
We'll correct it on the data

In [45]:
correct_sentence = 'the reason i left you is simple im gay'
word_count = len(correct_sentence.split())
data.loc[data["id"] == 125627, "word_count"] = word_count
data.loc[data["id"] == 125627, "normalized_text"] = correct_sentence

The only one missing is Gulliver Dark's sentence. After checking the epsiode, we have found no appearence of the character. This means that this is most likely a background song the character is singing and not really a line in the show.
We will delete this row 

In [ ]:
data = data[data['id'] != 156870]

# 4 Deleting Group characters 

With the previous processing step we realized that there are instances that don't identify charaters but groups of them. For the purpose of this assignemnt we want to focus only on the indviduall characters and therefore will delete all other instances, that augment the size of our dataset without providing any value

In [61]:
data['raw_character_text'].unique().tolist()


['Miss Hoover',
 'Lisa Simpson',
 'Edna Krabappel-Flanders',
 'Martin Prince',
 'Bart Simpson',
 'Landlady',
 'Nelson Muntz',
 'Terri/sherri',
 'Milhouse Van Houten',
 'Wendell Borton',
 'Kid Reporter',
 'Conductor',
 'BERGSTROM',
 'Homer Simpson',
 'Marge Simpson',
 'Ned Flanders',
 'Moe Szyslak',
 'Barney Gumble',
 'Dr. Julius Hibbert',
 "Husband Of Marge's Friend",
 'Maude Flanders',
 'Sophisticate #1',
 'Sophisticate #2',
 'Sophisticate #3',
 'Rev. Timothy Lovejoy',
 'Sultan',
 'Grampa Simpson',
 'Worm Man',
 'HELEN',
 'Gloria',
 'John',
 'Captain',
 'Otto Mann',
 'Convict',
 'Helen Lovejoy',
 'Customer',
 'Ticket Seller',
 'DIAMOND JOE',
 'Jimbo Jones',
 'Lost And Found Guy',
 'Radioactive Man',
 'Fallout Boy',
 'Announcer',
 'Convention Director',
 'BUDDY',
 'Dealer',
 'Young Patty',
 'Young Selma',
 'Older Patty',
 'Older Selma',
 'Mrs. Glick',
 'Adult Narrator',
 'Apu Nahasapeemapetilon',
 'Teller',
 'Eddie',
 'Lou',
 'Asa',
 'Soap Actress',
 'Soap Actor',
 'Mayor Joe Quimby',


# 5 Simplifying labels

# 3 Irrelevant Columns

there are a few columns that are also irrelevant for our use case.
* timestamp_in_ms -> we don't care about the moment in the episode the character speaks
* location_id -> we don't care about location
* raw_location_text -> same as the one above
* raw_text -> same as normalized text and spoken words, but adds noise such as emotions between brackets, which we are not interested in this assingment and make processing more complicated
* spoken_words -> it's repetitive having it's normalized version 

We can delete this columns


In [47]:
data.drop(columns = ['timestamp_in_ms','location_id','raw_location_text','raw_text','spoken_words'],inplace=True)

In [48]:
data

,id,episode_id,number,character_id,raw_character_text,normalized_text,word_count
0,9549,32,209,464.0,Miss Hoover,no actually it was a little of both sometimes ...,31
1,9550,32,210,9.0,Lisa Simpson,wheres mr bergstrom,3
2,9551,32,211,464.0,Miss Hoover,i dont know although id sure like to talk to h...,22
3,9552,32,212,9.0,Lisa Simpson,that life is worth living,5
4,9553,32,213,40.0,Edna Krabappel-Flanders,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...
158266,9544,32,204,464,Miss Hoover,im back,2
158267,9545,32,205,464,Miss Hoover,you see class my lyme disease turned out to be,10
158268,9546,32,206,464,Miss Hoover,psy-cho-so-ma-tic,1
158269,9547,32,207,119,Ralph Wiggum,does that mean you were crazy,6


# 4 Adding Season and number in season

we don't have a clear column indicating the season the epsiodes belong to, even though they can be deuced from the epsiode_id column, which indicates the epsiode without distingushing between seasons.
Since maaping this epsidoes to their respective seasons would requiere manually checking each one of them, we can use the dataset of the previous assingment that already has a defined 'season' column to create the column in our dataset

Since for the clean version of the dataset of the previous assignment we deleted all epsiodes of the last available season (it only had 4 episdes) to help our interests, we won't use this version for this assingment since we can use that information. We will read the raw one

In [49]:
data_seasons = pd.read_csv('../data/simpsons_episodes_raw.csv')[['season','number_in_season','number_in_series']]

In [50]:
episodes_daat1 = set(data['episode_id'])
episodes_data2 = set(data_seasons['number_in_series'])
episodes_daat1 - episodes_data2

set()

we see that all epsiodes in our script lines dataset are in the datset from the previous quarter

In [51]:
print(episodes_data2 - episodes_daat1)
len(episodes_data2 - episodes_daat1)

{550, 424, 569, 441, 570, 571, 572, 573, 575, 576, 577, 578, 579, 580, 581, 582, 583, 574, 447, 584, 585, 586, 587, 588, 589, 590, 591, 592, 593, 594, 595, 596, 597, 598, 599, 600}


36

However this is not true when inverting the order, the dataset from the previous assignment has 32 more episodes. The second source used on the previous assignment also has this same issue. We explored extra datasets such as this one: https://github.com/jcrodriguez1989/thesimpsons or https://github.com/rfordatascience/tidytuesday/blob/main/data/2025/2025-02-04/readme.md but to no success. No datasets contained the missing epsiodes

In [52]:
data = data.merge(data_seasons, left_on='episode_id', right_on='number_in_series', how='left')
data.drop(columns=['number_in_series'], inplace=True)
# put season before episode_id
cols = data.columns.tolist()
cols.insert(cols.index('episode_id'), cols.pop(cols.index('season')))
cols.insert(cols.index('episode_id'), cols.pop(cols.index('number_in_season')))
data = data[cols]
data

,id,season,number_in_season,episode_id,number,character_id,raw_character_text,normalized_text,word_count
0,9549,2,19,32,209,464.0,Miss Hoover,no actually it was a little of both sometimes ...,31
1,9550,2,19,32,210,9.0,Lisa Simpson,wheres mr bergstrom,3
2,9551,2,19,32,211,464.0,Miss Hoover,i dont know although id sure like to talk to h...,22
3,9552,2,19,32,212,9.0,Lisa Simpson,that life is worth living,5
4,9553,2,19,32,213,40.0,Edna Krabappel-Flanders,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...
132048,9544,2,19,32,204,464,Miss Hoover,im back,2
132049,9545,2,19,32,205,464,Miss Hoover,you see class my lyme disease turned out to be,10
132050,9546,2,19,32,206,464,Miss Hoover,psy-cho-so-ma-tic,1
132051,9547,2,19,32,207,119,Ralph Wiggum,does that mean you were crazy,6


# 5 Renamiming to keep columns undestandable

column number identifies the line within each, season. We find this name rather confusing, we'll change it to one that explains the variable better

In [54]:
data.rename(columns={'number': 'line_number_in_episode'}, inplace=True)  
data[data['episode_id'] ==  28]

,id,season,number_in_season,episode_id,line_number_in_episode,character_id,raw_character_text,normalized_text,word_count
123021,8143,2,15,28,4,1114,McBain,only your death,3
123022,8145,2,15,28,6,1114,McBain,meeting adjourned,2
123023,8150,2,15,28,11,1114,McBain,right now im thinking about holding another me...,10
123189,8205,2,15,28,66,31,Young Grampa,okay,1
123190,8207,2,15,28,68,31,Young Grampa,i promise,2
...,...,...,...,...,...,...,...,...,...
131130,8501,2,15,28,362,31,Grampa Simpson,i knew youd blow it,5
131131,8504,2,15,28,365,8,Bart Simpson,dad,1
131132,8505,2,15,28,366,2,Homer Simpson,what is it boy,4
131133,8506,2,15,28,367,8,Bart Simpson,i thought your car was really cool,7


In [55]:
data

,id,season,number_in_season,episode_id,line_number_in_episode,character_id,raw_character_text,normalized_text,word_count
0,9549,2,19,32,209,464.0,Miss Hoover,no actually it was a little of both sometimes ...,31
1,9550,2,19,32,210,9.0,Lisa Simpson,wheres mr bergstrom,3
2,9551,2,19,32,211,464.0,Miss Hoover,i dont know although id sure like to talk to h...,22
3,9552,2,19,32,212,9.0,Lisa Simpson,that life is worth living,5
4,9553,2,19,32,213,40.0,Edna Krabappel-Flanders,the polls will be open from now until the end ...,33
...,...,...,...,...,...,...,...,...,...
132048,9544,2,19,32,204,464,Miss Hoover,im back,2
132049,9545,2,19,32,205,464,Miss Hoover,you see class my lyme disease turned out to be,10
132050,9546,2,19,32,206,464,Miss Hoover,psy-cho-so-ma-tic,1
132051,9547,2,19,32,207,119,Ralph Wiggum,does that mean you were crazy,6


In [ ]:
data.to_csv('../data/simpsons_script_lines_cleaned.csv', index=False)